In [ ]:
# ============================================
# AMAZON REVIEWS + RoBERTa-Large (3-Class)
# ============================================

# pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn nltk

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

from datasets import Dataset, DatasetDict

# -----------------------------
# 1. Config
# -----------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "roberta-large"
MAX_LENGTH = 128
BATCH_SIZE = 8            # reduce to 4 if GPU memory is low
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 100          # requested
WARMUP_RATIO = 0.1
OUTPUT_DIR = "./roberta_large_amazon_output"

TRAIN_PATH = "Reviews.csv"   # change if needed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# -----------------------------
# 2. Load Data
# -----------------------------
df = pd.read_csv(TRAIN_PATH)

# Keep required columns
df = df[["Text", "Score"]].dropna()

# Remove duplicates
df = df.drop_duplicates(subset=["Text"]).reset_index(drop=True)

print("Original shape:", df.shape)

# -----------------------------
# 3. Label Mapping
# Score 1-2 = Negative (0)
# Score 3   = Neutral  (1)
# Score 4-5 = Positive (2)
# -----------------------------
def map_label(score):
    if score in [1, 2]:
        return 0
    elif score == 3:
        return 1
    else:
        return 2

df["label"] = df["Score"].apply(map_label)

# -----------------------------
# 4. Light Preprocessing
# -----------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["Text"] = df["Text"].apply(clean_text)

# Remove empty rows after cleaning
df = df[df["Text"].str.len() > 0].reset_index(drop=True)

print("After cleaning:", df.shape)
print(df["label"].value_counts())

# -----------------------------
# 5. Optional sampling
# RoBERTa-large on full Amazon data is expensive.
# Use this only if you want to limit training size.
# Comment it out for full training.
# -----------------------------
USE_SAMPLE = True
SAMPLE_SIZE = 120000   # adjust as needed

if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = (
        df.groupby("label", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 3), random_state=SEED))
          .reset_index(drop=True)
    )
    print("After stratified sampling:", df.shape)
    print(df["label"].value_counts())

# -----------------------------
# 6. Train/Val/Test Split
# -----------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# -----------------------------
# 7. Convert to HF Dataset
# -----------------------------
train_ds = Dataset.from_pandas(train_df[["Text", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["Text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["Text", "label"]], preserve_index=False)

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

# -----------------------------
# 8. Tokenizer
# -----------------------------
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["Text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

tokenized_ds = dataset.map(tokenize_function, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["Text"])
tokenized_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------------
# 9. Model
# -----------------------------
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

# -----------------------------
# 10. Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "precision_macro": macro_precision,
        "recall_macro": macro_recall,
        "f1_macro": macro_f1
    }

# -----------------------------
# 11. Training Arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

# -----------------------------
# 12. Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# -----------------------------
# 13. Train
# -----------------------------
trainer.train()

# -----------------------------
# 14. Evaluate on Test Set
# -----------------------------
test_results = trainer.evaluate(tokenized_ds["test"])
print("\nTest Results:")
for k, v in test_results.items():
    print(f"{k}: {v}")

# -----------------------------
# 15. Predictions
# -----------------------------
pred_output = trainer.predict(tokenized_ds["test"])
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

label_names = ["Negative", "Neutral", "Positive"]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# -----------------------------
# 16. Confusion Matrix
# -----------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - RoBERTa-large")
plt.tight_layout()
plt.show()

# -----------------------------
# 17. Save Best Model
# -----------------------------
trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))

print("\nBest model saved to:", os.path.join(OUTPUT_DIR, "best_model"))

In [ ]:
import re, random, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import spacy
nlp = spacy.load("en_core_web_sm")

SEED=42
random.seed(SEED); np.random.seed(SEED)

df = pd.read_csv("Reviews.csv")[["Text","Score"]].dropna().drop_duplicates("Text")

def map_label(s):
    return 0 if s in [1,2] else (1 if s==3 else 2)
df["label"] = df["Score"].apply(map_label)

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"<br\s*/?>"," ",t)
    t = re.sub(r"http\S+|www\S+"," ",t)
    t = re.sub(r"[^a-zA-Z\s']"," ",t)
    t = re.sub(r"\s+"," ",t).strip()
    doc = nlp(t)
    toks = [tok.lemma_ for tok in doc if not tok.is_stop and tok.is_alpha]
    return " ".join(toks)

df["clean"] = df["Text"].apply(clean_text)
df = df[df["clean"].str.len()>0]

# optional sampling (for speed)
df = df.sample(n=min(len(df),120000), random_state=SEED)

train, temp = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
val, test = train_test_split(temp, test_size=0.5, stratify=temp["label"], random_state=SEED)

print(train.shape, val.shape, test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train = tfidf.fit_transform(train["clean"])
X_val   = tfidf.transform(val["clean"])
X_test  = tfidf.transform(test["clean"])

y_train, y_val, y_test = train["label"], val["label"], test["label"]

models = {
    "SVM": LinearSVC(),
    "NB": MultinomialNB(),
    "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1),
    "XGB": XGBClassifier(tree_method='hist', eval_metric='mlogloss')
}

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(f"\n{name}:\n", classification_report(y_test, pred, digits=4))

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# build vocab
def build_vocab(texts, max_vocab=40000):
    cnt = Counter(" ".join(texts).split())
    vocab = {w:i+2 for i,(w,_) in enumerate(cnt.most_common(max_vocab))}
    vocab["<pad>"]=0; vocab["<unk>"]=1
    return vocab

vocab = build_vocab(train["clean"])

def encode(text, max_len=120):
    ids = [vocab.get(w,1) for w in text.split()][:max_len]
    return ids + [0]*(max_len-len(ids))

class TxtDS(Dataset):
    def __init__(self, df):
        self.X = [encode(t) for t in df["clean"]]
        self.y = df["label"].values
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i])

train_dl = DataLoader(TxtDS(train), batch_size=64, shuffle=True)
val_dl   = DataLoader(TxtDS(val), batch_size=64)

class BiGRU(nn.Module):
    def __init__(self, vocab_size, emb=128, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.gru = nn.GRU(emb, hid, batch_first=True, bidirectional=True)
        self.fc  = nn.Linear(hid*2, 3)
    def forward(self,x):
        x = self.emb(x)
        _,h = self.gru(x)
        h = torch.cat([h[-2],h[-1]], dim=1)
        return self.fc(h)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BiGRU(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for X,y in train_dl:
        X,y = X.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward(); opt.step()
    print("Epoch",epoch,"done")

In [ ]:
from gensim import corpora, models

texts = [t.split() for t in train["clean"][:20000]]
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(t) for t in texts]

lda = models.LdaModel(corpus, num_topics=8, id2word=dictionary, passes=5)

for i, topic in lda.print_topics(num_words=8):
    print(f"Topic {i}: {topic}")

In [ ]:
import torch
from datasets import Dataset
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def to_ds(df):
    return Dataset.from_pandas(df[["clean","label"]].rename(columns={"clean":"text"}))

train_ds, val_ds, test_ds = to_ds(train), to_ds(val), to_ds(test)

tok = RobertaTokenizerFast.from_pretrained("roberta-large")

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    acc = accuracy_score(p.label_ids, preds)
    p_,r_,f_,_ = precision_recall_fscore_support(p.label_ids, preds, average="weighted")
    return {"accuracy":acc,"f1":f_}

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=100,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=metrics,
    callbacks=[EarlyStoppingCallback(3)]
)

trainer.train()

res = trainer.evaluate(test_ds)
print(res)

In [ ]:
import shap
from lime.lime_text import LimeTextExplainer

# SHAP (sample)
explainer = shap.Explainer(lambda x: trainer.model(**tok(x, return_tensors="pt", padding=True)).logits.detach().numpy())
sample = test["clean"].iloc[:50].tolist()
shap_values = explainer(sample)

# LIME
lime_exp = LimeTextExplainer(class_names=["neg","neu","pos"])
i = 0
exp = lime_exp.explain_instance(test["clean"].iloc[i],
    lambda x: trainer.model(**tok(x, return_tensors="pt", padding=True)).logits.detach().numpy(),
    num_features=10)
exp.show_in_notebook()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - RoBERTa Large")
plt.show()

In [ ]:
#Amazon vs Yelp dataset

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
from lime.lime_text import LimeTextExplainer
import shap

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Utility: prediction function for SHAP/LIME
# ------------------------------------------------------------
def predict_proba_texts(texts, max_length=128, batch_size=16):
    trainer.model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(
            batch,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = trainer.model(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

# ============================================================
# PART 1 — ATTENTION HEATMAP EXTRACTION FROM RoBERTa
# ============================================================
# Example sentence; replace with your own sample if needed
sample_text = test["clean"].iloc[0]

trainer.model.eval()
enc = tok(sample_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    # Ensure attentions are returned
    outputs = trainer.model(**enc, output_attentions=True)

# attentions: tuple[num_layers] each of shape [batch, num_heads, seq_len, seq_len]
attentions = outputs.attentions
tokens = tok.convert_ids_to_tokens(enc["input_ids"][0].cpu().numpy())

# Average last layer across heads
last_layer_attn = attentions[-1][0].mean(dim=0).cpu().numpy()

plt.figure(figsize=(12, 10))
sns.heatmap(
    last_layer_attn,
    xticklabels=tokens,
    yticklabels=tokens,
    cmap="Greens",
    square=True,
    cbar=True
)
plt.title("RoBERTa Token-Level Attention Heatmap (Last Layer, Head-Averaged)", fontsize=16)
plt.xticks(rotation=90, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Optional: CLS attention contribution bar plot
# NOTE: RoBERTa uses <s> token instead of BERT [CLS], but this acts as the classification token.
cls_contrib = last_layer_attn[0]  # attention from <s> token to all tokens

plt.figure(figsize=(12, 5))
plt.bar(np.arange(len(tokens)), cls_contrib, color=sns.color_palette("crest", len(tokens)))
plt.xticks(np.arange(len(tokens)), tokens, rotation=90, fontsize=10)
plt.ylabel("Attention Weight", fontsize=13)
plt.title("Token Contribution to <s> (CLS-like) Representation", fontsize=16)
plt.tight_layout()
plt.show()

# ============================================================
# PART 2 — SHAP–LIME AGREEMENT PLOTS
# ============================================================
# Use a sample review
expl_text = test["clean"].iloc[1]

# ---------- LIME ----------
lime_explainer = LimeTextExplainer(class_names=["Negative", "Neutral", "Positive"])
lime_exp = lime_explainer.explain_instance(
    expl_text,
    predict_proba_texts,
    num_features=15,
    top_labels=1
)

lime_label = lime_exp.top_labels[0]
lime_map = dict(lime_exp.as_list(label=lime_label))

# ---------- SHAP ----------
# Use a text masker for token-level explanation
masker = shap.maskers.Text(tok)
shap_explainer = shap.Explainer(predict_proba_texts, masker)

# Single-sample explanation
shap_values = shap_explainer([expl_text])

# Token-level SHAP values for predicted class
pred_probs = predict_proba_texts([expl_text])[0]
pred_class = int(np.argmax(pred_probs))

# Extract SHAP tokens and values robustly
# shap_values.data[0] is tokenized pieces
try:
    shap_tokens = list(shap_values.data[0])
except Exception:
    shap_tokens = expl_text.split()

try:
    raw_vals = shap_values.values[0]
    # shape may be [tokens, classes] or [tokens]
    if raw_vals.ndim == 2:
        shap_token_vals = raw_vals[:, pred_class]
    else:
        shap_token_vals = raw_vals
except Exception:
    shap_token_vals = np.zeros(len(shap_tokens))

# Normalize token strings for rough matching
def norm_token(t):
    return str(t).replace("Ġ", "").strip().lower()

shap_df = pd.DataFrame({
    "token": [norm_token(t) for t in shap_tokens],
    "SHAP": np.abs(shap_token_vals)
})
shap_df = shap_df.groupby("token", as_index=False)["SHAP"].sum()

lime_df = pd.DataFrame({
    "token": [norm_token(t) for t in lime_map.keys()],
    "LIME": [abs(v) for v in lime_map.values()]
})
lime_df = lime_df.groupby("token", as_index=False)["LIME"].sum()

agree_df = pd.merge(shap_df, lime_df, on="token", how="inner")
agree_df = agree_df[agree_df["token"].str.len() > 0]

# Keep top common tokens
agree_df["importance"] = agree_df["SHAP"] + agree_df["LIME"]
agree_df = agree_df.sort_values("importance", ascending=False).head(15)

# ---------- Horizontal SHAP and LIME comparison ----------
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
plot_df = agree_df.sort_values("SHAP", ascending=True)

ax[0].barh(plot_df["token"], plot_df["SHAP"], color="royalblue", alpha=0.85)
ax[0].set_title("SHAP Token Importance", fontsize=16)
ax[0].set_xlabel("SHAP Score", fontsize=13)
ax[0].tick_params(axis='y', labelsize=12)

plot_df2 = agree_df.sort_values("LIME", ascending=True)
ax[1].barh(plot_df2["token"], plot_df2["LIME"], color="seagreen", alpha=0.85)
ax[1].set_title("LIME Token Importance", fontsize=16)
ax[1].set_xlabel("LIME Score", fontsize=13)
ax[1].tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

# ---------- SHAP–LIME agreement scatter ----------
plt.figure(figsize=(9, 7))
sc = plt.scatter(
    agree_df["SHAP"],
    agree_df["LIME"],
    s=(agree_df["importance"] * 2000) + 50,
    c=agree_df["importance"],
    cmap="coolwarm",
    alpha=0.8,
    edgecolor="black"
)

for _, row in agree_df.iterrows():
    plt.text(row["SHAP"] + 0.002, row["LIME"] + 0.002, row["token"], fontsize=10)

plt.xlabel("SHAP Score", fontsize=14)
plt.ylabel("LIME Score", fontsize=14)
plt.title("SHAP–LIME Agreement Plot", fontsize=16)
cbar = plt.colorbar(sc)
cbar.set_label("Combined Importance (SHAP + LIME)", fontsize=12)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ============================================================
# PART 3 — DOMAIN-SHIFT ANALYSIS (Amazon vs Yelp)
# ============================================================

# Predict Amazon
amazon_probs = predict_proba_texts(test["clean"].tolist())
amazon_pred = amazon_probs.argmax(axis=1)
amazon_true = test["label"].values

amazon_acc = accuracy_score(amazon_true, amazon_pred)
amazon_prec, amazon_rec, amazon_f1, _ = precision_recall_fscore_support(
    amazon_true, amazon_pred, average="weighted", zero_division=0
)

# Predict Yelp
yelp_probs = predict_proba_texts(yelp_test["clean"].tolist())
yelp_pred = yelp_probs.argmax(axis=1)
yelp_true = yelp_test["label"].values

yelp_acc = accuracy_score(yelp_true, yelp_pred)
yelp_prec, yelp_rec, yelp_f1, _ = precision_recall_fscore_support(
    yelp_true, yelp_pred, average="weighted", zero_division=0
)

# Uncertainty proxy = 1 - max prob mean
amazon_unc = 1 - amazon_probs.max(axis=1).mean()
yelp_unc = 1 - yelp_probs.max(axis=1).mean()

# ---------- Domain shift bar chart ----------
domains = ["Amazon", "Yelp"]
acc_vals = [amazon_acc * 100, yelp_acc * 100]
unc_vals = [amazon_unc * 100, yelp_unc * 100]

x = np.arange(len(domains))
width = 0.35

plt.figure(figsize=(10, 6))
bars1 = plt.bar(x - width/2, acc_vals, width, color="#7EC8E3", edgecolor="black", label="Accuracy (%)")
bars2 = plt.bar(x + width/2, unc_vals, width, color="#F7BFB4", edgecolor="black", label="Uncertainty (%)")

for b in bars1:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{b.get_height():.2f}",
             ha="center", fontsize=11)
for b in bars2:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{b.get_height():.2f}",
             ha="center", fontsize=11)

plt.xticks(x, domains, fontsize=13)
plt.ylabel("Percentage (%)", fontsize=14)
plt.title("Domain Shift Comparative Analysis", fontsize=16)
plt.legend(fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# ---------- Class-level cross-domain comparison ----------
# Assumes both domains use same 3 labels. If Yelp is binary, skip or adapt.
label_names = ["Negative", "Neutral", "Positive"]

def class_metrics(y_true, y_pred, average=None):
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0,1,2], zero_division=0)
    return p, r, f, s

if set(np.unique(yelp_true)).issubset({0,1,2}) and set(np.unique(amazon_true)).issubset({0,1,2}):
    a_p, a_r, a_f, _ = class_metrics(amazon_true, amazon_pred)
    y_p, y_r, y_f, _ = class_metrics(yelp_true, yelp_pred)

    idx = np.arange(3)
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))

    ax[0].bar(idx - 0.2, a_p, 0.4, label="Amazon", color=sns.color_palette("Blues")[3])
    ax[0].bar(idx + 0.2, y_p, 0.4, label="Yelp", color=sns.color_palette("Reds")[3])
    ax[0].set_xticks(idx); ax[0].set_xticklabels(label_names, rotation=15)
    ax[0].set_title("Precision by Class")

    ax[1].bar(idx - 0.2, a_r, 0.4, label="Amazon", color=sns.color_palette("Blues")[3])
    ax[1].bar(idx + 0.2, y_r, 0.4, label="Yelp", color=sns.color_palette("Reds")[3])
    ax[1].set_xticks(idx); ax[1].set_xticklabels(label_names, rotation=15)
    ax[1].set_title("Recall by Class")

    ax[2].bar(idx - 0.2, a_f, 0.4, label="Amazon", color=sns.color_palette("Blues")[3])
    ax[2].bar(idx + 0.2, y_f, 0.4, label="Yelp", color=sns.color_palette("Reds")[3])
    ax[2].set_xticks(idx); ax[2].set_xticklabels(label_names, rotation=15)
    ax[2].set_title("F1 by Class")

    for a in ax:
        a.legend(fontsize=10)
        a.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

# ---------- Confusion matrices ----------
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(amazon_true, amazon_pred), annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_title("Amazon Confusion Matrix")
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")

sns.heatmap(confusion_matrix(yelp_true, yelp_pred), annot=True, fmt="d", cmap="Purples", ax=ax[1])
ax[1].set_title("Yelp Confusion Matrix")
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

# ---------- Token polarity strength comparison (example on shared tokens) ----------
shared_tokens = ["taste","great","cold","weather","order","flavor"]

def token_strength(df_in, token_list, max_n=2000):
    # Sample reviews for speed
    sample_df = df_in.sample(n=min(len(df_in), max_n), random_state=42)
    vals = []
    for tk_ in token_list:
        idxs = sample_df["clean"].str.contains(rf"\b{tk_}\b", regex=True, na=False)
        if idxs.sum() == 0:
            vals.append(0.0)
        else:
            probs = predict_proba_texts(sample_df.loc[idxs, "clean"].tolist())
            # polarity strength = mean max confidence
            vals.append(float(probs.max(axis=1).mean()))
    return vals

amazon_strength = token_strength(test, shared_tokens)
yelp_strength = token_strength(yelp_test, shared_tokens)

plt.figure(figsize=(12, 6))
x = np.arange(len(shared_tokens))
plt.bar(x - 0.2, amazon_strength, width=0.4, label="Amazon", color="skyblue")
plt.bar(x + 0.2, yelp_strength, width=0.4, label="Yelp", color="salmon")
plt.xticks(x, shared_tokens, rotation=45, ha="right")
plt.ylabel("Polarity Strength")
plt.title("Amazon vs Yelp Token Polarity Strength Comparison", fontsize=16)
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# Optional: print summary
# ============================================================
print("\n=== Amazon ===")
print("Accuracy:", round(amazon_acc, 4), "Weighted F1:", round(amazon_f1, 4), "Uncertainty:", round(amazon_unc, 4))
print("\n=== Yelp ===")
print("Accuracy:", round(yelp_acc, 4), "Weighted F1:", round(yelp_f1, 4), "Uncertainty:", round(yelp_unc, 4))

In [ ]:
# ============================================================
# FULL EDA PIPELINE — AMAZON + YELP DATASETS
# ============================================================

# pip install pandas numpy matplotlib seaborn nltk wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from wordcloud import WordCloud
import nltk
nltk.download('punkt')

sns.set(style="whitegrid")

# ============================================================
# 1. LOAD DATA
# ============================================================

amazon_df = pd.read_csv("Reviews.csv")[["Text", "Score"]].dropna()
yelp_df   = pd.read_csv("yelp.csv")[["text", "stars"]].dropna()

amazon_df.columns = ["text", "score"]
yelp_df.columns   = ["text", "score"]

# ============================================================
# 2. LABEL CREATION
# ============================================================

def map_amazon(score):
    if score in [1,2]: return "Negative"
    elif score == 3: return "Neutral"
    else: return "Positive"

def map_yelp(score):
    if score <= 2: return "Negative"
    elif score == 3: return "Neutral"
    else: return "Positive"

amazon_df["sentiment"] = amazon_df["score"].apply(map_amazon)
yelp_df["sentiment"]   = yelp_df["score"].apply(map_yelp)

# ============================================================
# 3. BASIC DATA OVERVIEW
# ============================================================

print("Amazon shape:", amazon_df.shape)
print("Yelp shape:", yelp_df.shape)

print("\nAmazon class distribution:\n", amazon_df["sentiment"].value_counts())
print("\nYelp class distribution:\n", yelp_df["sentiment"].value_counts())

# ============================================================
# 4. TEXT CLEANING
# ============================================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

amazon_df["clean"] = amazon_df["text"].apply(clean_text)
yelp_df["clean"]   = yelp_df["text"].apply(clean_text)

# ============================================================
# 5. TEXT LENGTH ANALYSIS
# ============================================================

amazon_df["word_count"] = amazon_df["clean"].apply(lambda x: len(x.split()))
yelp_df["word_count"]   = yelp_df["clean"].apply(lambda x: len(x.split()))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.histplot(amazon_df["word_count"], bins=50, color="skyblue")
plt.title("Amazon Review Length Distribution")

plt.subplot(1,2,2)
sns.histplot(yelp_df["word_count"], bins=50, color="salmon")
plt.title("Yelp Review Length Distribution")

plt.tight_layout()
plt.show()

# ============================================================
# 6. SENTIMENT DISTRIBUTION COMPARISON
# ============================================================

fig, ax = plt.subplots(1,2, figsize=(12,5))

sns.countplot(x="sentiment", data=amazon_df, ax=ax[0], palette="Blues")
ax[0].set_title("Amazon Sentiment Distribution")

sns.countplot(x="sentiment", data=yelp_df, ax=ax[1], palette="Reds")
ax[1].set_title("Yelp Sentiment Distribution")

plt.tight_layout()
plt.show()

# ============================================================
# 7. WORD FREQUENCY ANALYSIS
# ============================================================

def get_top_words(text_series, n=20):
    words = " ".join(text_series).split()
    return Counter(words).most_common(n)

amazon_words = get_top_words(amazon_df["clean"])
yelp_words   = get_top_words(yelp_df["clean"])

# Convert to DataFrame
amazon_wdf = pd.DataFrame(amazon_words, columns=["word", "count"])
yelp_wdf   = pd.DataFrame(yelp_words, columns=["word", "count"])

# Plot
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
sns.barplot(x="count", y="word", data=amazon_wdf, palette="Blues")
plt.title("Top Words - Amazon")

plt.subplot(1,2,2)
sns.barplot(x="count", y="word", data=yelp_wdf, palette="Reds")
plt.title("Top Words - Yelp")

plt.tight_layout()
plt.show()

# ============================================================
# 8. BIGRAM ANALYSIS
# ============================================================

from nltk.util import ngrams

def get_top_ngrams(text_series, n=2, top_k=15):
    tokens = " ".join(text_series).split()
    ngrams_list = list(ngrams(tokens, n))
    return Counter(ngrams_list).most_common(top_k)

amazon_bigrams = get_top_ngrams(amazon_df["clean"], 2)
yelp_bigrams   = get_top_ngrams(yelp_df["clean"], 2)

def plot_ngrams(data, title, color):
    df = pd.DataFrame(data, columns=["ngram","count"])
    df["ngram"] = df["ngram"].apply(lambda x: " ".join(x))
    sns.barplot(x="count", y="ngram", data=df, color=color)
    plt.title(title)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plot_ngrams(amazon_bigrams, "Amazon Bigrams", "skyblue")

plt.subplot(1,2,2)
plot_ngrams(yelp_bigrams, "Yelp Bigrams", "salmon")

plt.tight_layout()
plt.show()

# ============================================================
# 9. WORD CLOUD VISUALIZATION
# ============================================================

def plot_wordcloud(text, title):
    wc = WordCloud(width=800, height=400, background_color="white").generate(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plot_wordcloud(" ".join(amazon_df["clean"]), "Amazon WordCloud")

plt.subplot(1,2,2)
plot_wordcloud(" ".join(yelp_df["clean"]), "Yelp WordCloud")

plt.tight_layout()
plt.show()

# ============================================================
# 10. SENTIMENT-WISE WORD DISTRIBUTION
# ============================================================

def sentiment_words(df, sentiment):
    return " ".join(df[df["sentiment"]==sentiment]["clean"])

for sentiment in ["Positive", "Negative"]:
    plt.figure(figsize=(10,4))
    plot_wordcloud(sentiment_words(amazon_df, sentiment), f"Amazon {sentiment}")
    plt.show()

# ============================================================
# 11. DOMAIN DIFFERENCE (IMPORTANT FOR PAPER)
# ============================================================

comparison = pd.DataFrame({
    "Metric": ["Avg Length", "Max Length"],
    "Amazon": [amazon_df["word_count"].mean(), amazon_df["word_count"].max()],
    "Yelp":   [yelp_df["word_count"].mean(), yelp_df["word_count"].max()]
})

print("\nDomain Comparison:\n", comparison)

# ============================================================
# 12. SAMPLE REVIEWS (FOR PAPER DISCUSSION)
# ============================================================

print("\nSample Amazon Reviews:\n", amazon_df["text"].sample(3).values)
print("\nSample Yelp Reviews:\n", yelp_df["text"].sample(3).values)